# Checkpoint Faithfulness Experiment Analysis

This notebook analyzes the results of the checkpoint faithfulness experiment.

**Research Question**: Does training meta-networks on diverse checkpoints (early, middle, final) lead to better **faithfulness** during unlearning compared to training only on final checkpoints?

**Hypothesis**: Training on early/middle checkpoints exposes the meta-network to weights that don't have the typical "SGD-optimized" look, expanding its trust region so predictions remain accurate when weights are modified significantly during unlearning.

In [ ]:
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats

# Set style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

## 1. Load Data

In [ ]:
# Path to results
RESULTS_DIR = Path("../experiments/checkpoint_faithfulness/results")

def load_all_results(results_dir: Path) -> pd.DataFrame:
    """Load all result CSVs from the results directory."""
    all_dfs = []
    for csv_path in results_dir.glob("*.csv"):
        if csv_path.name in ["summary.csv", "all_results.csv"]:
            continue
        df = pd.read_csv(csv_path)
        all_dfs.append(df)
    
    if not all_dfs:
        raise FileNotFoundError(f"No result CSVs found in {results_dir}")
    
    return pd.concat(all_dfs, ignore_index=True)

# Load data
df = load_all_results(RESULTS_DIR)
print(f"Loaded {len(df)} rows")
print(f"Datasets: {df['dataset'].unique()}")
print(f"Conditions: {df['condition'].unique()}")
print(f"Seeds: {df['seed'].unique()}")
df.head()

In [ ]:
# Parse JSON columns
def parse_list_column(series):
    """Parse a column containing JSON lists."""
    return series.apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

df['mean_diff_trajectory'] = parse_list_column(df['mean_diff_trajectory'])
df['original_accuracy'] = parse_list_column(df['original_accuracy'])
df['accuracy_after'] = parse_list_column(df['accuracy_after'])
df['init_pred'] = parse_list_column(df['init_pred'])
df['final_pred'] = parse_list_column(df['final_pred'])

## 2. Summary Statistics

In [ ]:
# Summary by dataset and condition
summary = df.groupby(['dataset', 'condition']).agg(
    n_samples=('model_idx', 'count'),
    mean_initial_mae=('initial_mae', 'mean'),
    std_initial_mae=('initial_mae', 'std'),
    mean_final_mae=('final_mae', 'mean'),
    std_final_mae=('final_mae', 'std'),
).reset_index()

summary

## 3. Final MAE Comparison

Compare final prediction error between final-only and multi-stage training.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot of final MAE by condition
ax = axes[0]
sns.boxplot(data=df, x='dataset', y='final_mae', hue='condition', ax=ax)
ax.set_title('Final MAE by Dataset and Condition')
ax.set_xlabel('Dataset')
ax.set_ylabel('Final MAE')
ax.legend(title='Condition')

# Violin plot
ax = axes[1]
sns.violinplot(data=df, x='dataset', y='final_mae', hue='condition', split=True, ax=ax)
ax.set_title('Final MAE Distribution')
ax.set_xlabel('Dataset')
ax.set_ylabel('Final MAE')
ax.legend(title='Condition')

plt.tight_layout()
plt.savefig('final_mae_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Statistical Tests

Test whether multi-stage training leads to significantly lower final MAE.

In [ ]:
def compute_statistical_tests(df: pd.DataFrame) -> pd.DataFrame:
    """Compute paired statistical tests between conditions."""
    results = []
    
    for dataset in df['dataset'].unique():
        df_dataset = df[df['dataset'] == dataset]
        
        final_only = df_dataset[df_dataset['condition'] == 'final-only']['final_mae'].values
        multi_stage = df_dataset[df_dataset['condition'] == 'multi-stage']['final_mae'].values
        
        # Use minimum length if different
        n = min(len(final_only), len(multi_stage))
        final_only = final_only[:n]
        multi_stage = multi_stage[:n]
        
        # Paired t-test
        t_stat, t_pvalue = stats.ttest_rel(final_only, multi_stage)
        
        # Wilcoxon signed-rank test (non-parametric)
        w_stat, w_pvalue = stats.wilcoxon(final_only, multi_stage)
        
        # Effect size (Cohen's d)
        diff = final_only - multi_stage
        cohens_d = diff.mean() / diff.std()
        
        results.append({
            'dataset': dataset,
            'n_pairs': n,
            'mean_final_only': final_only.mean(),
            'mean_multi_stage': multi_stage.mean(),
            'mean_difference': diff.mean(),
            't_statistic': t_stat,
            't_pvalue': t_pvalue,
            'wilcoxon_statistic': w_stat,
            'wilcoxon_pvalue': w_pvalue,
            'cohens_d': cohens_d,
        })
    
    return pd.DataFrame(results)

test_results = compute_statistical_tests(df)
test_results

## 5. Faithfulness Trajectory Analysis

Analyze how mean_diff evolves over unlearning steps.

In [ ]:
def plot_mean_diff_trajectory(df: pd.DataFrame, dataset: str, max_steps: int = 1000):
    """Plot mean_diff trajectory for both conditions."""
    df_dataset = df[df['dataset'] == dataset]
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    for condition in ['final-only', 'multi-stage']:
        df_cond = df_dataset[df_dataset['condition'] == condition]
        
        # Stack all trajectories
        trajectories = []
        for traj in df_cond['mean_diff_trajectory']:
            if len(traj) > 0:
                # Truncate to max_steps
                trajectories.append(traj[:max_steps])
        
        if not trajectories:
            continue
        
        # Pad to same length
        max_len = max(len(t) for t in trajectories)
        padded = np.array([t + [np.nan] * (max_len - len(t)) for t in trajectories])
        
        # Compute mean and std
        mean_traj = np.nanmean(padded, axis=0)
        std_traj = np.nanstd(padded, axis=0)
        
        steps = np.arange(len(mean_traj))
        ax.plot(steps, mean_traj, label=condition, linewidth=2)
        ax.fill_between(steps, mean_traj - std_traj, mean_traj + std_traj, alpha=0.2)
    
    ax.set_xlabel('Unlearning Step')
    ax.set_ylabel('Mean Diff (MAE)')
    ax.set_title(f'Faithfulness Trajectory - {dataset.upper()}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Add threshold lines
    for thresh in [0.05, 0.1, 0.15]:
        ax.axhline(y=thresh, color='red', linestyle='--', alpha=0.3, label=f'threshold={thresh}')
    
    plt.tight_layout()
    return fig

# Plot for each dataset
for dataset in df['dataset'].unique():
    fig = plot_mean_diff_trajectory(df, dataset)
    plt.savefig(f'trajectory_{dataset}.png', dpi=150, bbox_inches='tight')
    plt.show()

## 6. Unfaithfulness Step Analysis

Find at which step mean_diff exceeds various thresholds.

In [ ]:
def get_unfaithfulness_step(trajectory: list, threshold: float) -> int:
    """Find first step where mean_diff exceeds threshold."""
    for step, val in enumerate(trajectory):
        if val > threshold:
            return step
    return -1  # Never exceeded

# Compute unfaithfulness steps for different thresholds
thresholds = [0.05, 0.10, 0.15, 0.20]

for thresh in thresholds:
    df[f'unfaithfulness_step_{int(thresh*100)}'] = df['mean_diff_trajectory'].apply(
        lambda x: get_unfaithfulness_step(x, thresh)
    )

# Summary of unfaithfulness steps
unfaith_summary = df.groupby(['dataset', 'condition']).agg({
    'unfaithfulness_step_5': ['mean', lambda x: (x == -1).mean()],
    'unfaithfulness_step_10': ['mean', lambda x: (x == -1).mean()],
    'unfaithfulness_step_15': ['mean', lambda x: (x == -1).mean()],
}).reset_index()

unfaith_summary.columns = ['_'.join(col).strip('_') for col in unfaith_summary.columns.values]
unfaith_summary

## 7. Distance vs Faithfulness

Analyze relationship between weight space distance and prediction error.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, dataset in enumerate(df['dataset'].unique()[:4]):
    ax = axes[idx // 2, idx % 2]
    df_dataset = df[df['dataset'] == dataset]
    
    for condition in ['final-only', 'multi-stage']:
        df_cond = df_dataset[df_dataset['condition'] == condition]
        ax.scatter(
            df_cond['distance_travelled'],
            df_cond['final_mae'],
            alpha=0.3,
            label=condition,
            s=10
        )
    
    ax.set_xlabel('Distance Travelled')
    ax.set_ylabel('Final MAE')
    ax.set_title(f'{dataset.upper()}')
    ax.legend()

plt.suptitle('Distance Travelled vs Final MAE', fontsize=14)
plt.tight_layout()
plt.savefig('distance_vs_faithfulness.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Conclusions

In [ ]:
print("="*60)
print("EXPERIMENT SUMMARY")
print("="*60)

for _, row in test_results.iterrows():
    dataset = row['dataset']
    diff = row['mean_difference']
    pval = row['t_pvalue']
    d = row['cohens_d']
    
    sig = "*" if pval < 0.05 else ""
    sig += "*" if pval < 0.01 else ""
    sig += "*" if pval < 0.001 else ""
    
    direction = "higher" if diff > 0 else "lower"
    
    print(f"\n{dataset.upper()}:")
    print(f"  Final-only MAE {direction} by {abs(diff):.4f} (p={pval:.4f}{sig})")
    print(f"  Cohen's d = {d:.3f} ({'small' if abs(d) < 0.5 else 'medium' if abs(d) < 0.8 else 'large'} effect)")

print("\n" + "="*60)
print("Significance: * p<0.05, ** p<0.01, *** p<0.001")